# Multi-View Patch Extraction for 3D BBox Assets (REFRESHED)

**FORCE REFRESH: Updated at 2026-03-21 09:05**

This notebook extracts and saves multi-view image patches for 3D bounding box assets. Core helpers are in `utils/patch_extraction.py`.

### Timestamp System Explanation

The dataset uses **Unit: Microseconds ($\mu s = 10^{-6}$ seconds)** for all internal timestamps.

**Why they don't exactly match 10Hz/30Hz multiples:**
Autonomous vehicle sensors (LIDAR, Cameras, RADAR) have different capture timings and slight jitter. 
- **Labels**: Annotated periodically (e.g., at LIDAR sweep times).
- **Cameras**: Have their own exposure times. 

To ensure perfect 3D alignment, we must find the **Actual capture time** $t_{camera}$ for a given frame and then **interpolate** the ego-vehicle pose (`egomotion`) for that exact microsecond. The timestamps in this clip (e.g., `17,700,212`) refer to time since the beginning of the sequence.

### Reprojection Logic ($World \rightarrow Rig \rightarrow Camera \rightarrow Pixel$)

To accurately project a 3D box defined in labels onto the camera image at time $t$:
1. **World Frame (t)**: The labels define `center_x,y,z` in the World/Map coordinate system. 
2. **Rig Frame (t)**: We transform the object's world corners to the local ego-vehicle (Rig) frame using the ego-pose at time $t$:  
   $P_{rig} = T_{rig \leftarrow world}(t) \cdot P_{world}$
3. **Camera Frame**: We transform local Rig coordinates to the specific camera sensor frame using known extrinsics: 
   $P_{cam} = T_{cam \leftarrow rig} \cdot P_{rig}$
4. **Image Plane**: Finally, we use the camera's intrinsic matrix ($K$) to project the 3D-camera points into 2D pixel coordinates: 
   $uv = K \cdot P_{cam}$

In [ ]:
%load_ext autoreload
%autoreload 2

import pathlib
import random
from collections import defaultdict

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from scipy.spatial.transform import Rotation

import physical_ai_av
from physical_ai_av.utils import tf

from utils.patch_extraction import get_corners, project, get_patch_rect, draw_bbox_3d

## Parameters

In [ ]:
clip_id = "3c16fed6-60c0-4cab-9b24-ac1af36c9c99"
min_cameras = 2
min_patch_size = 200
output_dir = pathlib.Path("./data/output_patches")
visibility_threshold = 0.99

In [ ]:
avdi = physical_ai_av.PhysicalAIAVDatasetInterface(cache_dir="/tmp/huggingface")

extrinsics = avdi.get_clip_feature(clip_id, feature=avdi.features.CALIBRATION.SENSOR_EXTRINSICS, maybe_stream=True)
intrinsics = avdi.get_clip_feature(clip_id, feature=avdi.features.CALIBRATION.CAMERA_INTRINSICS, maybe_stream=True)
egomotion = avdi.get_clip_feature(clip_id, feature=avdi.features.LABELS.EGOMOTION, maybe_stream=True)

camera_ids = list(intrinsics.camera_models.keys())
print(f"Cameras: {camera_ids}")

## Load and Map 3D Bounding Boxes

In [ ]:
# Load obstacles (this usually returns a dict for .zip features)
bbox_raw = avdi.get_clip_feature(clip_id, feature=avdi.features.LABELS.OBSTACLE_OFFLINE, maybe_stream=True)

if isinstance(bbox_raw, dict):
    # Pick 'obstacle' key if multiple exist in the zip
    key = next((k for k in bbox_raw if "obstacle" in k.lower()), list(bbox_raw.keys())[0])
    bbox_df = bbox_raw[key]
else:
    bbox_df = bbox_raw

print(f"Loaded {len(bbox_df)} annotations. Columns: {list(bbox_df.columns)}")

# Column constants for matching OBSTACLE_OFFLINE
COL_ID = "track_id"
COL_TS = "timestamp_us"
COL_REF_TS = "reference_frame_timestamp_us"

## Visualization Check (One frame, all cameras)

Run this cell to verify that the 3D bboxes are correctly reprojected and aligned with the images before extraction.

In [ ]:
# Pick a random timestamp with at least one obstacle
sample_ts = int(bbox_df.sample(1)[COL_TS].iloc[0])

# Select ALL obstacles within 100 ms (100000 us) window around sample_ts
window_us = 100000
rows = bbox_df[np.abs(bbox_df[COL_TS] - sample_ts) < window_us]

print(f"Visualizing {len(rows)} obstacles within {window_us/1000}ms of target label timestamp {sample_ts//1000}ms")

fig, axes = plt.subplots(len(camera_ids), 1, figsize=(15, 5*len(camera_ids)))
for i, cam_id in enumerate(camera_ids):
    ax = axes[i] if len(camera_ids) > 1 else axes
    try:
        feat = next(f for f in avdi.features.CAMERA.ALL if cam_id in f or f.endswith(cam_id))
        reader = avdi.get_clip_feature(clip_id, feature=feat, maybe_stream=True)
        
        # ALIGNMENT: Find the closest image frame to our target label timestamp
        img_stack, actual_ts_array = reader.decode_images_from_timestamps(np.array([sample_ts], dtype=np.int64))
        actual_ts = int(actual_ts_array[0])
        img = Image.fromarray(img_stack[0])
        
        for _, row in rows.iterrows():
            corners = get_corners(row)
            # project now uses actual_ts (the camera capture time) to look up ego-pose precisely
            uv, mask, in_front = project(corners, row[COL_TS], actual_ts, cam_id, egomotion, extrinsics, intrinsics, image_size=img.size)
            draw_bbox_3d(img, uv, in_front)
        
        ax.imshow(img)
        ax.set_title(f"{cam_id} (image_ts: {actual_ts//1000}ms, target_ts: {sample_ts//1000}ms)")
    except Exception as e:
        ax.text(0.5, 0.5, f"Error: {e}", ha='center', va='center')
    ax.axis('off')
plt.tight_layout()
plt.show()

## Process Best Patches per (Object, Camera) Pair

In [7]:
from tqdm.auto import tqdm

print("Optimizing extraction metadata...")
cam_info = {}
for cam_id in tqdm(camera_ids, desc="Checking camera resolutions"):
    if cam_id not in extrinsics.sensor_poses: continue
    norm = cam_id.lower().replace("-", "_")
    feat = next((f for f in avdi.features.CAMERA.ALL if norm in f.lower()), None)
    if not feat: continue
    reader = avdi.get_clip_feature(clip_id, feature=feat, maybe_stream=True)
    cam_info[cam_id] = {"feat": feat, "w": reader.stream.width, "h": reader.stream.height}

print("Processing best patches per (object, camera) pair...")
best_patches = defaultdict(dict)
obj_dims = {}
grouped = list(bbox_df.groupby(COL_ID))

for obj_id, obj_df in tqdm(grouped, desc="Analyzing object visibility"):
    obj_dims[str(obj_id)] = {"length": round(float(obj_df.iloc[0]["size_x"]), 2), "width": round(float(obj_df.iloc[0]["size_y"]), 2), "height": round(float(obj_df.iloc[0]["size_z"]), 2)}
    for cam_id, info in cam_info.items():
        w_img, h_img = info["w"], info["h"]
        max_area, best_cand = -1, None
        
        for _, row in obj_df.iterrows():
            ts = int(row[COL_TS])
            if ts < egomotion.timestamps.min() or ts > egomotion.timestamps.max(): continue
            
            # Note: For patch selection, we use the annotation time ts.
            corners = get_corners(row)
            uv, mask, in_front = project(corners, ts, ts, cam_id, egomotion, extrinsics, intrinsics, image_size=(w_img, h_img))
            if mask.sum() / len(corners) < visibility_threshold: continue
            
            r = get_patch_rect(uv, mask)
            if not r: continue
            x0, y0, x1, y1 = max(0, r[0]), max(0, r[1]), min(w_img, r[2]), min(h_img, r[3])
            w, h = x1-x0, y1-y0
            if w < min_patch_size or h <s min_patch_size: continue
            if w*h > max_area:
                max_area = w*h
                best_cand = {"ts": ts, "bbox": (x0, y0, x1, y1)}

        if best_cand: best_patches[obj_id][cam_id] = best_cand

qualified = {oid: d for oid, d in best_patches.items() if len(d) >= min_cameras}
print(f"\nSummary: {len(grouped)} total objects, {len(best_patches)} visible, {len(qualified)} qualified for reconstruction.")

Optimizing extraction metadata...


Checking camera resolutions: 100%|██████████| 7/7 [00:02<00:00,  2.97it/s]


Processing best patches per (object, camera) pair...


Analyzing object visibility: 100%|██████████| 109/109 [00:44<00:00,  2.45it/s]


Summary: 109 total objects, 11 visible, 6 qualified for reconstruction.


## Save All Patches

In [8]:
print("Loading video and saving patches...")
req_ts = defaultdict(set)
for d in qualified.values():
    for c, cand in d.items(): req_ts[c].add(cand["ts"])

saved = 0
for cam_id, tss in tqdm(req_ts.items(), desc="Saving patches"):
    feat = cam_info[cam_id]["feat"]
    reader = avdi.get_clip_feature(clip_id, feature=feat, maybe_stream=True)
    imgs, actual_ts = reader.decode_images_from_timestamps(np.array(sorted(list(tss)), dtype=np.int64))
    frames = {int(actual_ts[i]): imgs[i] for i in range(len(actual_ts))}
    
    for obj_id, d in qualified.items():
        if cam_id not in d: continue
        ts, (x0, y0, x1, y1) = d[cam_id]["ts"], d[cam_id]["bbox"]
        
        # ALIGNMENT: Re-project for the EXACT frame being saved
        mapped = np.array(list(frames.keys()))
        best_ts = int(mapped[np.argmin(np.abs(mapped - ts))])
            
        obj_path = output_dir / clip_id / str(obj_id) / "multi_view_imgs"
        obj_path.mkdir(parents=True, exist_ok=True)
        Image.fromarray(frames[best_ts][y0:y1, x0:x1]).save(obj_path / f"{cam_id}_{best_ts}.png")
        dims_path = output_dir / clip_id / str(obj_id) / "dims.json"
        if not dims_path.exists():
            import json
            with open(dims_path, "w") as f:
                json.dump(obj_dims[str(obj_id)], f)
        saved += 1

print(f"\nSuccess: Saved {saved} patches to {output_dir.resolve()}")

Loading video and saving patches...


Saving patches: 100%|██████████| 7/7 [00:07<00:00,  1.09s/it]


Success: Saved 18 patches to /home/ubuntu/workspace/asset_generation/data/output_patches
